# AMC Project — Google Colab Runner

RadioML 2016.10A üzerinde CNN2 baseline + CBAM-eklenmiş model karşılaştırması.

**Tek seferlik kurulum:**
1. Bu notebook'u Colab'da aç (**Runtime → Change runtime type → GPU** seç).
2. `amc_project/` klasörünün Google Drive'da olduğundan emin ol: `MyDrive/amcprojects/amc_project/`.
3. `RML2016.10a_dict.pkl` (~640 MB) dosyasının `data/` içinde olduğunu doğrula.
4. Aşağıdaki hücreleri sırayla çalıştır.

**Toplam tahmini GPU süresi:** ~50-60 dk (AMP sayesinde).

## 1. Drive'ı bağla & GPU'yu doğrula

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
# Proje yolunu Drive'daki gerçek konumla eşle
import os
PROJECT_DIR = '/content/drive/MyDrive/amcprojects/amc_project'
assert os.path.isdir(PROJECT_DIR), f'Proje klasörü bulunamadı: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('cwd =', os.getcwd())
!ls

## 2. Bağımlılıkları yükle

Colab'da PyTorch + numpy + matplotlib zaten kurulu; sadece eksikleri tamamlıyoruz.

In [ ]:
!pip install -q optuna tensorboard seaborn

In [ ]:
import torch
print('torch  =', torch.__version__)
print('cuda   =', torch.cuda.is_available())
print('device =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Veri ön-işleme (tek seferlik, ~2-3 dk)

Pickle'ı yükler, normalize eder, 60/20/20 stratified split yapar ve `data/processed/` altına `.npy` olarak yazar.
Drive'da kalır — sonraki notebook session'larında bu hücreyi atlayabilirsin (zaten varsa veri tekrar üretilmez).

In [ ]:
# Eğer processed dosyalar zaten varsa atla, yoksa üret
from pathlib import Path
if (Path('data/processed') / 'X_train.npy').exists():
    print('[setup] processed dosyalar zaten var, ön-işleme atlanıyor.')
else:
    !python data_loader.py

## 4. EDA — constellation diyagramları (opsiyonel, ilk seferde)

In [ ]:
!python visualize.py

In [ ]:
# Üretilen figürleri inline göster
from IPython.display import Image, display
from pathlib import Path
for p in sorted(Path('figures/eda').glob('*.png')):
    print(p.name)
    display(Image(str(p)))

## 5. TensorBoard'u inline aç (eğitim sırasında izle)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 6. Baseline'ı eğit (CNN2)

AMP (mixed precision) açık → T4'te epoch ~15-25 sn.
Toplam ~10-15 dk + early stopping.

**Not:** Aynı modeli ikinci kez eğitirsen yeni bir timestamp'li run klasörü oluşur, eskisi silinmez.

In [ ]:
!python train.py --model cnn2 --epochs 30 --num_workers 0 --tag baseline

## 7. CBAM-enhanced modeli eğit

CBAM ekstra parametreler ekler, biraz daha yavaş ama hâlâ AMP ile hızlı.

In [ ]:
!python train.py --model cnn2_cbam --epochs 30 --num_workers 0 --tag cbam_v1

## 8. Değerlendir & figürleri üret

In [ ]:
# En son baseline run'ını ve CBAM run'ını otomatik bul
from pathlib import Path
runs = sorted(Path('runs').glob('*'))
baseline_runs = [r for r in runs if r.name.startswith('cnn2_') and 'cbam' not in r.name]
cbam_runs = [r for r in runs if 'cbam' in r.name]
BASELINE = baseline_runs[-1] if baseline_runs else None
CBAM = cbam_runs[-1] if cbam_runs else None
print('baseline ckpt:', BASELINE)
print('cbam ckpt    :', CBAM)
assert BASELINE is not None and CBAM is not None, 'Önce 6. ve 7. hücreleri çalıştırman lazım.'

In [ ]:
!python evaluate.py --ckpt {BASELINE}/best.pt

In [ ]:
!python evaluate.py --ckpt {CBAM}/best.pt

In [ ]:
# Üretilen figürleri inline göster
from IPython.display import Image, display
from pathlib import Path
for run_dir in (BASELINE, CBAM):
    fig_dir = Path('figures') / run_dir.name
    print(f'=== {run_dir.name} ===')
    for p in sorted(fig_dir.glob('*.png')):
        print(p.name)
        display(Image(str(p)))

## 9. Baseline vs CBAM — TAM karşılaştırma

`compare.py` üretir:
- `figures/comparison/acc_vs_snr_overlay.png` — iki eğri tek grafikte + Δaccuracy bar (sunumun ana figürü)
- `figures/comparison/confusion_side_by_side.png` — iki confusion matrix yan yana
- `figures/comparison/qam_confusion_bar.png` — QAM16↔QAM64 confusion azalması
- `figures/comparison/summary.md` — rapor için hazır markdown tablo
- `figures/comparison/summary.json` — tüm metrikler

In [ ]:
!python compare.py

In [ ]:
# Karşılaştırma figürlerini inline göster
from IPython.display import Image, Markdown, display
from pathlib import Path

cmp_dir = Path('figures/comparison')
for p in sorted(cmp_dir.glob('*.png')):
    print(p.name)
    display(Image(str(p)))

# Özet markdown'ı göster
summary_md = cmp_dir / 'summary.md'
if summary_md.exists():
    display(Markdown(summary_md.read_text()))

## 11. Hiperparametre Optimizasyonu (Optuna)

Bu adim ~1.5 saat GPU surer. **Onemli:** Laptop uyku moduna girmesin, sarja takili olsun, tarayici sekmesi acik kalsin.

`tune.py` yapar:
1. SQLite-tabanli persistent study basla (kopusa dayanikli)
2. 15 trial Bayesian optimization (TPE), her biri 8 epoch + median pruning
3. En iyi config'le 30 epoch tam egitim
4. Final ckpt'yi `runs/cnn2_cbam_..._tuned/best.pt` altinda kaydet

**Optimize edilen parametreler:**
- `lr`: 1e-4 — 5e-3 (log scale)
- `dropout`: {0.3, 0.4, 0.5, 0.6}
- `weight_decay`: {1e-5, 1e-4, 1e-3}
- CBAM `reduction`: {4, 8, 16}

**Objective:** SNR ≥ 0 dB validation accuracy (proposal'la uyumlu).

In [ ]:
!python tune.py --n_trials 15 --epochs_per_trial 8 --final_epochs 30

## 12. Optuna sonuçlarını görselleştir

Trial başına performansı, hangi hiperparametrelerin önemli olduğunu görüyoruz.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

results = json.loads(Path('optuna_studies/results.json').read_text())
print(f"Toplam trial: {results['n_trials']}")
print(f"Tamamlanan  : {results['n_complete']}")
print(f"Pruned      : {results['n_pruned']}")
print(f"En iyi value (val high-SNR acc): {results['best_value']:.4f}")
print(f"En iyi params: {results['best_params']}")
if 'final_test_acc' in results:
    print(f"Final test accuracy: {results['final_test_acc']:.4f}")

In [ ]:
# Trial-by-trial gorsellestir
import optuna
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

study = optuna.load_study(study_name='cnn2_cbam_search', storage='sqlite:///optuna_studies/study.db')

# 1. Optimization history
ax1 = plot_optimization_history(study)
plt.tight_layout()
plt.savefig('figures/optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()

# 2. Param importances
try:
    ax2 = plot_param_importances(study)
    plt.tight_layout()
    plt.savefig('figures/optuna_importances.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Importance plot atlandi: {e}')

# 3. Parallel coordinate
try:
    ax3 = plot_parallel_coordinate(study)
    plt.tight_layout()
    plt.savefig('figures/optuna_parallel.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Parallel plot atlandi: {e}')

print('Optuna gorsellestirmeleri figures/ altina kaydedildi.')

In [ ]:
!python rerun_top3.py --epochs 30 --patience 10

In [ ]:
# Top-3 rerun sonucundan en iyi modeli bul
import json
from pathlib import Path

top3_results = json.loads(Path('optuna_studies/top3_results.json').read_text())
BEST_TUNED = Path(top3_results[0]['run_dir'])
print(f"En iyi tuned model: {BEST_TUNED}")
print(f"Test acc: {top3_results[0]['test_acc']:.4f}")
print(f"Trial num: {top3_results[0]['trial_num']}")
print(f"Params: {top3_results[0]['params']}")

In [ ]:
!python evaluate.py --ckpt {BEST_TUNED}/best.pt

In [ ]:
# 1. Tuned vs Baseline (en geniş karşılaştırma — sunum ana figürü)
!python compare.py --baseline {BASELINE} --cbam {BEST_TUNED}

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path
cmp_dir = Path('figures/comparison')
for p in sorted(cmp_dir.glob('*.png')):
    print(p.name)
    display(Image(str(p)))
summary_md = cmp_dir / 'summary.md'
if summary_md.exists():
    display(Markdown(summary_md.read_text()))

## 13. Tuned modeli evaluate et + baseline ile karşılaştır

Tuned CBAM ckpt'sini değerlendirip ORİJİNAL baseline ile karşılaştır.

In [ ]:
# Tuned run'ini bul
from pathlib import Path
runs = sorted(Path('runs').glob('*'))
tuned_runs = [r for r in runs if 'tuned' in r.name]
TUNED = tuned_runs[-1] if tuned_runs else None
print('tuned ckpt:', TUNED)
assert TUNED is not None, 'tune.py final egitimi bulunamadi.'

In [ ]:
!python evaluate.py --ckpt {TUNED}/best.pt

In [ ]:
# 3-yonlu karsilastirma: Baseline vs CBAM-default vs CBAM-tuned
!python compare.py --baseline {BASELINE} --cbam {TUNED}

In [ ]:
# Tuned vs default CBAM karsilastirmasi (asil iyilesmeyi gormek icin)
!python compare.py --baseline {CBAM} --cbam {TUNED}

In [ ]:
# Karsilastirma figurlerini goster
from IPython.display import Image, Markdown, display
from pathlib import Path

cmp_dir = Path('figures/comparison')
print("=" * 60)
print("KARSILASTIRMA FIGURLERI (en son compare.py cagrisindan):")
print("=" * 60)
for p in sorted(cmp_dir.glob('*.png')):
    print(p.name)
    display(Image(str(p)))

summary_md = cmp_dir / 'summary.md'
if summary_md.exists():
    display(Markdown(summary_md.read_text()))

In [ ]:
from pathlib import Path
import json

runs = sorted(Path('runs').glob('*'))
baseline_runs = [r for r in runs if r.name.startswith('cnn2_') and 'cbam' not in r.name]
default_cbam_runs = [r for r in runs if 'cbam_v1' in r.name]

BASELINE = baseline_runs[-1] if baseline_runs else None
CBAM = default_cbam_runs[-1] if default_cbam_runs else None

# Top-1 modelini Optuna sonuçlarından al
top3_results = json.loads(Path('optuna_studies/top3_results.json').read_text())
BEST_TUNED = Path(top3_results[0]['run_dir'])

print('BASELINE   :', BASELINE)
print('CBAM       :', CBAM)
print('BEST_TUNED :', BEST_TUNED)
print(f'\nBest tuned test acc: {top3_results[0]["test_acc"]:.4f}')
print(f'Best tuned params  : {top3_results[0]["params"]}')

In [ ]:
!python ensemble.py

In [ ]:
!python compare.py \
  --baseline {BASELINE} \
  --cbam_metrics figures/ensemble_top3 \
  --baseline_label "CNN2 (baseline)" \
  --cbam_label "Top-3 Ensemble" \
  --out_subdir comparison_ensemble_vs_baseline

In [ ]:
!python compare.py \
  --baseline {CBAM} \
  --cbam {BEST_TUNED} \
  --baseline_label "CNN2+CBAM (default)" \
  --cbam_label "CNN2+CBAM (Optuna tuned)" \
  --out_subdir comparison_tuned_vs_default

In [ ]:
!python compare.py \
  --baseline {BEST_TUNED} \
  --cbam_metrics figures/ensemble_top3 \
  --baseline_label "CNN2+CBAM (tuned)" \
  --cbam_label "Top-3 Ensemble" \
  --out_subdir comparison_ensemble_vs_tuned

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path

dirs = [
    'comparison_ensemble_vs_baseline',
    'comparison_tuned_vs_default',
    'comparison_ensemble_vs_tuned',
]

for sub in dirs:
    d = Path('figures') / sub
    if not d.exists():
        continue
    print(f"\n{'='*60}\n{sub}\n{'='*60}")
    for p in sorted(d.glob('*.png')):
        print(p.name)
        display(Image(str(p)))
    summary_md = d / 'summary.md'
    if summary_md.exists():
        display(Markdown(summary_md.read_text()))

In [ ]:
import shutil
import time
from pathlib import Path

# Backup hedef dizini
backup_root = Path('/content/drive/MyDrive/amc_backup_final')
backup_root.mkdir(parents=True, exist_ok=True)

# Tüm önemli klasörleri kopyala
for src_name in ['runs', 'figures', 'optuna_studies']:
    src = Path(src_name)
    if not src.exists():
        print(f"  ✗ {src_name} bulunamadı")
        continue
    dst = backup_root / src_name
    if dst.exists():
        shutil.rmtree(dst)
    print(f"  Kopyalanıyor: {src_name} -> {dst}")
    shutil.copytree(src, dst)
    print(f"  ✓ {src_name} kopyalandı")

# Önemli dosyaları da kopyala
for src_name in ['RESULTS.md', 'README.md', 'config.py', 'data_loader.py',
                 'models.py', 'train.py', 'tune.py', 'rerun_top3.py',
                 'ensemble.py', 'compare.py', 'evaluate.py', 'visualize.py',
                 'requirements.txt', 'colab_setup.md']:
    src = Path(src_name)
    if src.exists():
        shutil.copy2(src, backup_root / src_name)
        print(f"  ✓ {src_name}")

print(f"\n✅ Backup tamamlandı: {backup_root}")
print("Boyut kontrolü:")
!du -sh /content/drive/MyDrive/amc_backup_final

## 10. (Opsiyonel) Checkpoint'leri Drive'a yedekle

In [ ]:
import shutil, time
from pathlib import Path
backup = Path('/content/drive/MyDrive/amc_backup') / time.strftime('%Y%m%d-%H%M%S')
backup.mkdir(parents=True, exist_ok=True)
for run_dir in (BASELINE, CBAM, BEST_TUNED):
    if run_dir is None: continue
    src = run_dir / 'best.pt'
    if src.exists():
        shutil.copy2(src, backup / f'{run_dir.name}_best.pt')
        print('copied', src, '->', backup)
print('backup dir:', backup)